In [1]:
import os
import pathlib
import sys

here = pathlib.Path.cwd()
ROOT = here.parents[2] if here.name == "day03" else here
os.chdir(ROOT)
SANDBOX = ROOT / "sandbox" / "w5" / "day03"

print("프로젝트 루트  :", ROOT)

if str(ROOT / 'backend') not in sys.path:
    sys.path.insert(0, str(ROOT / 'backend'))

SAMPLES = ROOT / "sandbox" / "w5" / "day01" / "samples"

프로젝트 루트  : c:\hw\hw\python_basic\hanwha-agent


**샘플 파일 조회**

In [2]:
if not SAMPLES.is_dir():
    print('재료 폴더가 없습니다')
else:
    all_files = [p for p in SAMPLES.rglob('*') if p.is_file()]
    top_files = [p for p in SAMPLES.iterdir() if p.is_file()]
    sub_files = [p for p in (SAMPLES / '_formats').rglob('*') if p.is_file()]
    today = sorted((p.name for p in SAMPLES.glob('DOC-HR-014_*_v2.0.*')))
    print(f'재료 파일 : {len(all_files)}개')
    print(f'  samples 바로 아래 : {len(top_files)}개 · _formats 안 : {len(sub_files)}개')
    print('오늘 읽을 세 형식 :')
    for name in today:
        print(' ', name)

재료 파일 : 30개
  samples 바로 아래 : 24개 · _formats 안 : 6개
오늘 읽을 세 형식 :
  DOC-HR-014_국내출장_여비_규정_v2.0.docx
  DOC-HR-014_국내출장_여비_규정_v2.0.hwpx
  DOC-HR-014_국내출장_여비_규정_v2.0.pdf


**로컬 파서가 리턴해주는 데이터 확인**

In [3]:
from app.rag.local_parsers import parse_local
TARGET = next(SAMPLES.glob('DOC-HR-014_*_v2.0.docx'))
doc = parse_local(TARGET)
print(f'블록 {len(doc.blocks)} · 표 {doc.table_count} · 쪽 {doc.page_count}')

블록 233 · 표 7 · 쪽 1


**파싱한 블럭 하나가 어떤식으로 데이터를 갖고 있는지 확인**

In [4]:
for i, b in enumerate(doc.blocks[:5]):
    head = b.text[:36] + ('…' if len(b.text) > 36 else '')
    print(f'[{i:2d}] {b.kind} | {b.locator} | {head}')
first_table = next((b for b in doc.blocks if b.kind == '표'))
print()
print('첫 표 블록 : locator=' + first_table.locator)
print(' ', first_table.text[:60])

[ 0] 조항 | DOC-HR-014_국내출장_여비_규정_v2.0 | 일반
[ 1] 조항 | DOC-HR-014_국내출장_여비_규정_v2.0 | 국내출장 여비 규정
[ 2] 조항 | DOC-HR-014_국내출장_여비_규정_v2.0 | v2.0
[ 3] 조항 | DOC-HR-014_국내출장_여비_규정_v2.0 | 문서번호  DOC-HR-014
[ 4] 조항 | DOC-HR-014_국내출장_여비_규정_v2.0 | 시행일    2025-07-01

첫 표 블록 : locator=표1
  | 버전 | 구분 | 시행일 | 주요 내용 | 작성 | 검토 | 승인 |
|---|---|---|---|--
